# Disaster Recommender System — Fine-Tuning Integration
This notebook retains your retrieval pipeline (embeddings + FAISS + geospatial re-ranking) and replaces the RAG + Gemini generation with a LoRA fine-tuned GPT-Neo 2.7B generator.

In [1]:
!pip install transformers accelerate peft bitsandbytes faiss-cpu sentence-transformers datasets

In [2]:
import os
import json
import pandas as pd
import numpy as np
import faiss
import logging
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, pipeline
from peft import LoraConfig, get_peft_model
import bitsandbytes as bnb
import matplotlib.pyplot as plt
logging.basicConfig(level=logging.INFO)

2025-05-20 13:25:38.027156: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747747538.051384     151 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747747538.058949     151 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## 1. Data Loading & Preprocessing

In [3]:
df = pd.read_csv('/kaggle/input/disaster-recommendation-dataset/disasterRecommendation_dataset.csv')
df = df.drop(columns=[c for c in df.columns if 'Unnamed' in c])
df = df.rename(columns={'latitude':'lat','longitude':'lon'})
df = df.dropna(subset=['lat','lon','disaster_type'])
df['text'] = df.apply(lambda row: (
    f"Location: {row['location']}. Type: {row['disaster_type']}. "
    f"Urgency: {row['urgency_level']}. Shelter: {row['shelters']}. "
    f"Routes: {row['evacuation_routes']}. Precautions: {row['precautionary_measures']}."
), axis=1)
df.head()

,disaster_type,location,urgency_level,source,shelters,evacuation_routes,emergency_contact,first_aid,precautionary_measures,lat,lon,text
0,flood,Lahore,high,UN OCHA,Model Town Community Center,Lahore Ring Road to GT Road,999,"Protective helmet, emergency blanket",Identify storm shelter,31.549717,74.343617,Location: Lahore. Type: flood. Urgency: high. ...
1,tornado,Gwadar,low,UN OCHA,Gwadar Port Relief Tent,Makran Coastal Highway,999,"Emergency blanket, antiseptic wipes",Reinforce windows with shutters,25.126227,62.325227,Location: Gwadar. Type: tornado. Urgency: low....
2,tornado,Faisalabad,low,NDMA,D Ground Community Hall,Canal Road to Dijkot Road,1123,"Injury splints, antiseptic solution",Identify storm shelter,31.449520,73.134120,Location: Faisalabad. Type: tornado. Urgency: ...
3,cyclone,Gwadar,medium,UN OCHA,Gwadar Port Relief Tent,Makran Coastal Highway,999,"Flashlight, whistle, first aid kit",Identify safe spots in rooms,25.127123,62.326123,Location: Gwadar. Type: cyclone. Urgency: medi...
4,earthquake,Muzaffarabad,high,WHO,Red Crescent Camp,Kashmir Highway to Neelum Road,1122,"Tourniquets, antiseptic solution",Stay away from riverbanks after heavy rain,34.370656,73.470656,Location: Muzaffarabad. Type: earthquake. Urge...


## 2. Embedding & FAISS Indexing

In [4]:
embed_model = SentenceTransformer('paraphrase-mpnet-base-v2')
texts = df['text'].tolist()
embeddings = embed_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
faiss.write_index(index, 'faiss_index.bin')
meta = df[['text','lat','lon','location','disaster_type','shelters','evacuation_routes','precautionary_measures','urgency_level']]
meta.to_csv('metadata.csv', index=False)
print(f'Indexed {index.ntotal} documents')

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Indexed 50000 documents


## 3. Retrieval + Geospatial Re-ranking Functions

In [5]:
def haversine(lat1, lon1, lat2, lon2):
    R=6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1,lon1,lat2,lon2])
    dlat, dlon = lat2-lat1, lon2-lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def retrieve(query, qlat, qlon, qtype, k=5, radius=50):
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    D,I = index.search(q_emb, 300)
    c = meta.iloc[I[0]].copy()
    c['semantic'] = D[0]
    c['dist'] = haversine(qlat,qlon,c.lat.values,c.lon.values)
    near = c[c['dist']<=radius]
    if len(near)==0:
        near = c.nsmallest(k,'dist')
    urgency_map={'high':0.15,'medium':0.08,'low':0.0}
    adapt = {'earthquake':1,'tornado':0.7,'cyclone':0.7,'flood':0.5,'landslide':0.6}
    near['adapt']=near['disaster_type'].map(lambda x: adapt.get(x,0.3))
    near['dist_score']=1/(1+(near.dist/15))
    near['urg']=near['urgency_level'].map(urgency_map)
    near['score']=0.6*near.dist_score+0.2*near.adapt+0.1*near.semantic+near.urg
    top=near.sort_values('score',ascending=False).head(k)
    return top

## 4. Prompt Builder

In [6]:
def build_prompt(top, qlat, qlon, qtype, loc_name):
    shelters=''.join([f"- {r.shelters} ({r.location}, {r.dist:.1f}km)\n" for r in top.itertuples()])
    routes=''.join([f"- {r}\n" for r in top.evacuation_routes.unique()])
    prec=''.join([f"- {r}\n" for r in top.precautionary_measures.unique()])
    return (
        f"Emergency plan for {qtype} at ({qlat},{qlon}) [{loc_name}]:\n"
        f"Available shelters:\n{shelters}\n"
        f"Evacuation routes:\n{routes}\n"
        f"Safety precautions:\n{prec}\n"
        "Format as 4 numbered immediate actions under 200 words."
    )

## 5. Prepare Q&A Pairs for Fine-Tuning

In [7]:
qa=[]
for _,r in df.iterrows():
    p=f"User at ({r.lat},{r.lon}) experiencing {r.disaster_type}. What shelters and routes?"
    c=f"Shelter: {r.shelters}. Routes: {r.evacuation_routes}. Precautions: {r.precautionary_measures}."
    qa.append({'prompt':p,'completion':c})
with open('qa.jsonl','w') as f:
    for x in qa: f.write(json.dumps(x)+'\n')
ds=load_dataset('json', data_files={'train':'qa.jsonl'})['train'].train_test_split(0.1)
ds

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 45000
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 5000
    })
})

## 6. LoRA Fine-Tuning GPT-Neo 2.7B

In [8]:
pip install -U bitsandbytes

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [9]:
from transformers import BitsAndBytesConfig, DataCollatorForLanguageModeling, Trainer, TrainingArguments, AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
import torch
model_name = "EleutherAI/gpt-neo-125M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
quant_config = BitsAndBytesConfig(load_in_4bit=True)
device_map = {"": torch.cuda.current_device()}
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map=device_map
)

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05
)
model = get_peft_model(model, lora_config)

def tok_fn(ex):
    inp = tokenizer(
        ex["prompt"],
        truncation=True,
        padding="max_length",
        max_length=64
    )
    tgt = tokenizer(
        ex["completion"],
        truncation=True,
        padding="max_length",
        max_length=64
    )
    return {
        "input_ids": inp.input_ids,
        "attention_mask": inp.attention_mask,
        "labels": tgt.input_ids
    }

td = ds.map(tok_fn, batched=True)
td.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./ft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_first_step=True,
    logging_steps=10,
    save_steps=500,
    max_steps=1000,         
    fp16=True,
    report_to="none",        
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=td["train"],
    eval_dataset=td["test"],
    data_collator=data_collator,
)

trainer.train()
model.save_pretrained("./ft/lora_adapter")
tokenizer.save_pretrained("./ft/lora_adapter")


model.safetensors:   0%|          | 0.00/526M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/bitsandbytes/nn/modules.py:451: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(


Step,Training Loss
1,3.328600
10,0.000000
20,0.000000
30,0.000000
40,0.000000
50,0.000000
60,0.000000
70,0.000000
80,0.000000
90,0.000000


('./ft/lora_adapter/tokenizer_config.json',
 './ft/lora_adapter/special_tokens_map.json',
 './ft/lora_adapter/vocab.json',
 './ft/lora_adapter/merges.txt',
 './ft/lora_adapter/added_tokens.json',
 './ft/lora_adapter/tokenizer.json')

## 7. Inference with Fine-Tuned Model

In [10]:
tok_ft=AutoTokenizer.from_pretrained('./ft/lora_adapter')
model_ft=AutoModelForCausalLM.from_pretrained('./ft/lora_adapter',load_in_4bit=True,device_map='auto')
gen = pipeline('text-generation', model=model_ft, tokenizer=tok_ft, max_length=200)
def example():
    q='Experiencing an earthquake at Fish Farm Faisalabad.'
    lat,lon=31.45,73.13; typ='earthquake'; loc='Fish Farm Faisalabad'
    top=retrieve(q,lat,lon,typ)
    prompt=build_prompt(top,lat,lon,typ,loc)
    print(gen(prompt, num_return_sequences=1)[0]['generated_text'])
example()

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Device set to use cuda:1


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Emergency plan for earthquake at (31.45,73.13) [Fish Farm Faisalabad]:
Available shelters:
- Railway Market Relief Tent (Faisalabad, 0.4km)
- Railway Market Relief Tent (Faisalabad, 0.4km)
- Chenab Club Shelter (Faisalabad, 0.4km)
- Railway Market Relief Tent (Faisalabad, 0.4km)
- Railway Market Relief Tent (Faisalabad, 0.4km)

Evacuation routes:
- Canal Road to Dijkot Road
- GT Road to Jaranwala Link
- Sargodha Road

Safety precautions:
- Reinforce garage doors
- Stay in interior rooms
- Identify storm shelter

Format as 4 numbered immediate actions under 200 words.!!!!!!!!!!!!!!!!!!!!!
